# Lab 5 — Interactive Plots with Plotly & a Complete EDA Walkthrough

**DS331 · Introduction to Data Science**

Two parts today:

**Part A — Plotly Express:** interactive charts you can hover, zoom, and filter. Wonderful for *exploring* data and for presentations.

**Part B — The full EDA pipeline, end to end:** everything from Labs 1–4 applied to one dataset in one place. **This is the template for your report.**

1. [Why interactive plots?](#1)
2. [Plotly Express basics — scatter](#2)
3. [The Plotly gallery: bar, histogram, box, line](#3)
4. [Animation — data over time](#4)
5. [Hierarchies — sunburst](#5)
6. [Part B: complete EDA walkthrough](#6)
7. [Your report checklist](#7)
8. [Try it yourself ✏️](#8)

<a id="1"></a>
## 1. Why interactive plots?

**Plotly Express** (`px`) follows the same logic as seaborn — `px.someplot(df, x=..., y=..., color=...)` — but produces **interactive** figures:

- **hover** over a point → see its exact values (and any columns you add);
- **zoom/pan** → inspect crowded regions;
- **click legend entries** → hide/show groups.

When to use which: **matplotlib/seaborn** for static report figures (PDF, print); **plotly** for exploration, dashboards, and HTML/slides. Knowing both makes you flexible.

Plotly is pre-installed in Colab — just import it:

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

gapminder = px.data.gapminder()     # life expectancy, GDP, population by country & year
gapminder.head()

<a id="2"></a>
## 2. Plotly Express basics — scatter

The same scatter logic as seaborn, with `color=` instead of `hue=`. Run the cell, then **hover over points, zoom with the mouse wheel, and click legend entries**:

In [ ]:
g2007 = gapminder[gapminder["year"] == 2007]

fig = px.scatter(
    g2007,
    x="gdpPercap", y="lifeExp",
    color="continent",
    size="pop", size_max=55,            # bubble size = population
    hover_name="country",               # bold name in the tooltip
    log_x=True,                         # GDP is heavily skewed -> log scale
    title="Wealth vs life expectancy, 2007 (bubble size = population)",
    labels={"gdpPercap": "GDP per capita (log scale)", "lifeExp": "life expectancy (years)"},
)
fig.show()

This single chart encodes **five** variables: x, y, color, size, and the hover name. Note `labels={...}` — the plotly way to set axis labels, and `log_x=True` for skewed data (compare with what we learned about skew in Lab 2!).

<a id="3"></a>
## 3. The Plotly gallery

The same chart types you know — now interactive. Watch how the interface stays identical across plot types.

In [ ]:
# Bar: average life expectancy per continent in 2007
avg_life = g2007.groupby("continent", as_index=False)["lifeExp"].mean()

fig = px.bar(avg_life, x="continent", y="lifeExp", color="continent",
             title="Average life expectancy by continent (2007)",
             labels={"lifeExp": "life expectancy (years)"})
fig.show()

In [ ]:
# Histogram: with a marginal box plot on top — two views in one figure
fig = px.histogram(g2007, x="lifeExp", nbins=25, marginal="box",
                   title="Distribution of life expectancy (2007)",
                   labels={"lifeExp": "life expectancy (years)"})
fig.show()

In [ ]:
# Box: distribution per group; hover a box to read its quartiles exactly
fig = px.box(g2007, x="continent", y="gdpPercap", color="continent", log_y=True,
             title="GDP per capita by continent (2007)")
fig.show()

In [ ]:
# Line: one line per country — click legend entries to isolate countries
asia5 = gapminder[gapminder["country"].isin(
    ["Bangladesh", "India", "China", "Japan", "Korea, Rep."])]

fig = px.line(asia5, x="year", y="lifeExp", color="country", markers=True,
              title="Life expectancy over time — five Asian countries",
              labels={"lifeExp": "life expectancy (years)"})
fig.show()

<a id="4"></a>
## 4. Animation — data over time

Plotly's party trick: `animation_frame=` adds a play button. The famous "Gapminder" animation in one function call — **press ▶ below the chart**:

In [ ]:
fig = px.scatter(
    gapminder,
    x="gdpPercap", y="lifeExp",
    color="continent", size="pop", size_max=55,
    hover_name="country",
    log_x=True,
    animation_frame="year",             # one frame per year
    animation_group="country",          # points move smoothly between frames
    range_y=[25, 90],                   # fix the axes so motion is comparable
    title="World development, 1952–2007",
)
fig.show()

Fixing the axis ranges (`range_y=...`) matters: if the axes rescaled each frame, the motion would be meaningless.

<a id="5"></a>
## 5. Hierarchies — sunburst

When categories nest (continent → country), a **sunburst** shows part-of-whole structure at two levels. **Click a continent slice to zoom into it; click the center to zoom out:**

In [ ]:
fig = px.sunburst(g2007, path=["continent", "country"], values="pop",
                  title="World population 2007: continent → country (click to zoom!)")
fig.show()

A few more `px` functions worth knowing for your own dataset: `px.violin`, `px.density_heatmap`, `px.scatter_matrix` (interactive pairplot), `px.treemap`, `px.choropleth` (world maps).

> ⚠️ **Plotly in reports:** interactive charts cannot be pasted into a PDF. For a written report, either export a static image (`fig.write_image("plot.png")`, needs `pip install kaleido`) or submit the notebook itself (interactivity survives in Colab/HTML).

---

<a id="6"></a>
## 6. Part B — The complete EDA walkthrough 🚢

Everything from all five labs, in the exact order your report should follow. We use the Titanic dataset and keep each step short — your report fills in the depth.

### Step 0 — The questions

EDA without questions is wandering. Write down 2–4 before you start. Ours:

1. What factors are associated with surviving the disaster?
2. Did passenger class matter?
3. Did age and sex matter ("women and children first")?

### Step 1 — Load and inspect *(Lab 1)*

In [ ]:
df = sns.load_dataset("titanic")

print("shape:", df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

**Notes from inspection:** 891 rows × 15 columns. Numeric: `age`, `fare`, `sibsp`, `parch`. Categorical: `sex`, `pclass`, `embarked`… Missing data in `age` (177), `deck` (688), `embarked` (2).

### Step 2 — Clean *(Lab 2)*

Decisions, with reasons — exactly what your report should state:

In [ ]:
df = df.drop(columns=["deck"])                    # 77% missing -> not usable
df = df.dropna(subset=["embarked"])               # only 2 rows -> safe to drop
df["age"] = df["age"].fillna(df["age"].median())  # 20% missing -> fill with median

print("remaining missing values:", df.isna().sum().sum())
print("shape after cleaning:", df.shape)

### Step 3 — Engineer features *(Lab 3)*

In [ ]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = df["family_size"] == 1
df["age_group"] = pd.cut(df["age"], bins=[0, 12, 19, 35, 60, 100],
                         labels=["child", "teen", "young adult", "adult", "senior"])

df[["family_size", "is_alone", "age_group"]].head()

### Step 4 — Univariate exploration *(Lab 4)* — each variable alone

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

sns.countplot(data=df, x="survived", ax=axes[0])
axes[0].set_title("Survival (0 = died, 1 = survived)")

sns.countplot(data=df, x="pclass", ax=axes[1])
axes[1].set_title("Passenger class")

sns.histplot(data=df, x="age", bins=25, ax=axes[2])
axes[2].set_title("Age distribution")

fig.tight_layout()
plt.show()

**Observations:** ~38% survived; 3rd class was the largest group; ages center on 20–35 (the spike at the median age, 28, is our imputation from Step 2 — an honest report points this out!).

### Step 5 — Bivariate exploration — variables *against the question*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

sns.barplot(data=df, x="pclass", y="survived", ax=axes[0])
axes[0].set_title("Survival rate by class")

sns.barplot(data=df, x="sex", y="survived", ax=axes[1])
axes[1].set_title("Survival rate by sex")

sns.barplot(data=df, x="age_group", y="survived", ax=axes[2])
axes[2].set_title("Survival rate by age group")
axes[2].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

In [ ]:
# The same finding as a table — reports should mix plots AND numbers
df.groupby(["pclass", "sex"])["survived"].mean().round(2).unstack()

**Observations:** survival drops sharply with class (1st ≈ 63%, 3rd ≈ 24%); women survived far more often than men (74% vs 19%); children fared better than seniors. First-class women survived at ~97%, third-class men at ~14% — the combination of factors is dramatic.

### Step 6 — Multivariate view + correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=df, x="pclass", y="age", hue="survived", ax=axes[0])
axes[0].set_title("Age by class and survival")

corr = df[["survived", "age", "fare", "family_size", "pclass"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Correlations")

fig.tight_layout()
plt.show()

In [ ]:
# One interactive figure for exploration: every passenger, hoverable
fig = px.scatter(df, x="age", y="fare", color="survived",
                 hover_data=["sex", "pclass", "family_size"],
                 opacity=0.6, title="Each passenger: age vs fare, colored by survival")
fig.show()

### Step 7 — Write the conclusions

The report ends with answers to the Step-0 questions, in plain language:

> 1. **Class mattered enormously:** 1st-class survival ≈ 63%, 3rd-class ≈ 24% (corr. with `survived`: −0.34).
> 2. **Sex was the strongest factor:** 74% of women vs 19% of men survived.
> 3. **Age mattered at the extremes:** children ≈ 58%, seniors ≈ 27% — consistent with "women and children first".
> 4. **Limitations:** age was imputed for 20% of passengers; correlation does not establish causation.

Note the structure of every conclusion: *claim + number + (where relevant) caveat.*

<a id="7"></a>
## 7. Your report checklist ✅

Structure your report exactly like Part B:

1. **Introduction** — your dataset, where it came from, and 2–4 questions.
2. **Loading & inspection** — shape, dtypes, `describe()`, missing-value table.
3. **Cleaning** — every decision *with its reason*.
4. **Feature engineering** — 2–4 motivated features.
5. **Univariate analysis** — distributions of the key variables.
6. **Bi-/multivariate analysis** — plots that address your questions; mix figures and numbers.
7. **Conclusions** — answer each question with claim + number + caveat.

Common mistakes that cost marks:

- ❌ plots with no title/axis labels, or with no interpretation sentence below them
- ❌ cleaning silently ("I dropped some rows") — say *what, how many, and why*
- ❌ a wall of 20 plots with no questions driving them — fewer, chosen plots win
- ❌ claiming causation from correlation
- ❌ notebook that doesn't run top to bottom (always test: *Runtime → Restart and run all*)

<a id="8"></a>
## 8. Try it yourself ✏️

**Exercise 1.** With `px.scatter` on the 2007 gapminder data, plot population (`log_x=True`) vs life expectancy, colored by continent. Hover around: which *country* is the small-population outlier with very low life expectancy?

**Exercise 2.** Build an interactive box plot of `lifeExp` per continent for **1952** and another for **2007**. Which continent improved the most?

**Exercise 3.** Pick any 3 countries and draw their `gdpPercap` over time with `px.line`. Write three observations (claim + number each).

**Exercise 4 (mini-report).** Run the full Part-B pipeline on the `tips` dataset: one question, inspect, (light) clean, engineer one feature (e.g. tip percentage), two plots, one conclusion. This is a dress rehearsal for your report!

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4 — mini report

---
🎉 **That's the full toolkit:** loading → inspecting → cleaning → engineering → visualizing → concluding. Apply it to your assigned dataset, keep your questions in front of you, and explain every figure. Good luck with your reports!